# Evaluate a Bedrock Model with Inspect AI

## What is Inspect AI?

[Inspect AI](https://inspect.ai-safety-institute.org.uk/) is an open-source framework for LLM evaluations by the UK AI Safety Institute. It provides a standardized way to define benchmarks, run them against any model, and score the results — making evaluations reproducible and comparable across models.

## What is the SageMaker Inspect AI Container?

The [SageMaker Inspect AI container](https://docs.aws.amazon.com/nova/latest/userguide/inspect-ai-sagemaker-eval.html) runs LLM evaluations as a SageMaker Training Job. It uses the Training Job as managed compute for orchestration (not for actual model training) — your model runs separately on its own inference infrastructure.

## How This Notebook Works

This notebook shows two approaches to evaluate Amazon Nova (or any Bedrock-hosted model):

| Approach | How it runs | Best for |
|----------|------------|----------|
| **Local SDK** | `inspect eval` CLI on your machine | Fast iteration, development, small runs |
| **Container Job** | SageMaker Training Job | Scalable, hands-off, production runs |

Both approaches call the same Bedrock model and run the same benchmarks — they differ only in where the orchestration happens.

## Prerequisites

1. **AWS credentials configured** — via `aws configure`, environment variables, or IAM role
   - [Configuring the AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/cli-chap-configure.html)
2. **Bedrock model access enabled** — [Enable in Bedrock console](https://console.aws.amazon.com/bedrock/home#/modelaccess)
3. **Python 3.10+**

**Time to complete:** ~10 minutes | **Cost:** Bedrock per-token charges + `ml.m5.large` (~$0.13/hr) for Approach 2

---

## Approach 1: Local SDK (Quick Evaluation)

Run evaluations directly on your machine using the Inspect AI CLI. Results in seconds for small sample sizes.

In [ ]:
%pip install "inspect-ai>=0.3.220" inspect-evals --quiet

### Verify Environment

In [ ]:
import subprocess, shutil, boto3

# Check inspect CLI
if shutil.which("inspect"):
    result = subprocess.run(["inspect", "--version"], capture_output=True, text=True)
    print(f"\u2713 Inspect AI: {result.stdout.strip()}")
else:
    print("\u2717 Inspect CLI not found. Restart your terminal after pip install.")

# Check AWS credentials
try:
    identity = boto3.client("sts").get_caller_identity()
    print(f"\u2713 AWS credentials (Account: {identity['Account']})")
except Exception as e:
    print(f"\u2717 AWS credentials not configured: {e}")

### Run Evaluation

We run MMLU-Pro (5 samples) against Nova 2 Lite on Bedrock. MMLU-Pro is an advanced multiple-choice reasoning benchmark with 12K samples.

**Key parameters:**
- `--model`: The Bedrock model to evaluate
- `--limit`: Number of samples (remove for full dataset)
- `--max-connections`: Parallel requests to Bedrock

In [ ]:
# Run MMLU-Pro (5 samples) against Nova 2 Lite on Bedrock
!inspect eval inspect_evals/mmlu_pro \
    --model bedrock/us.amazon.nova-2-lite-v1:0 \
    -M region_name=us-east-1 \
    --limit 5 \
    --max-connections 4 \
    --max-retries 3 \
    --display plain

### View Results

The Inspect AI viewer opens a web UI showing aggregate scores, per-sample results, and error analysis.

In [ ]:
!inspect view

---

## Approach 2: Container Job (Scalable Evaluation)

Run the same evaluation as a SageMaker Training Job. The container:
1. Downloads your config and benchmarks from S3
2. Installs benchmark dependencies
3. Calls Bedrock for each evaluation sample
4. Publishes results to S3 incrementally

No GPU needed — the `ml.m5.large` orchestrator instance only coordinates the eval.

In [ ]:
%pip install "boto3>=1.35" "sagemaker>=3.0.0" pyyaml --quiet

### Setup: IAM Role and S3 Bucket

The training job needs a role with Bedrock, S3, and SageMaker permissions.

In [ ]:
import boto3
import json
import os
import time
import yaml

REGION = "us-east-1"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
S3_BUCKET = f"inspectlens-eval-{ACCOUNT_ID}"
ROLE_NAME = "InspectLensEvalRole"
ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
IMAGE_URI = "763104351884.dkr.ecr.us-east-1.amazonaws.com/sagemaker-inspect-ai:latest"

# Create IAM role
iam = boto3.client("iam")
trust_policy = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "sagemaker.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
try:
    iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    print(f"\u2713 Created role: {ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    print(f"\u2713 Role exists: {ROLE_NAME}")
policies = ["arn:aws:iam::aws:policy/AmazonSageMakerFullAccess", "arn:aws:iam::aws:policy/AmazonS3FullAccess", "arn:aws:iam::aws:policy/AmazonBedrockFullAccess"]
for arn in policies:
    iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=arn)
# Add MLflow permissions (for optional tracking)
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="MLflowAccess", PolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Action": ["sagemaker:DescribeMlflowTrackingServer", "sagemaker:CreatePresignedMlflowTrackingServerUrl", "sagemaker-mlflow:*"], "Resource": f"arn:aws:sagemaker:{REGION}:{ACCOUNT_ID}:mlflow-tracking-server/*"}]}))
time.sleep(10)

# Create S3 bucket
s3 = boto3.client("s3", region_name=REGION)
try:
    s3.create_bucket(Bucket=S3_BUCKET) if REGION == "us-east-1" else s3.create_bucket(Bucket=S3_BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})
    print(f"\u2713 Created bucket: {S3_BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"\u2713 Bucket exists: {S3_BUCKET}")

### Create and Upload a Benchmark

A benchmark is a Python file with an `@task`-decorated function that defines:
- **Dataset** — the evaluation samples
- **Solver** — how the model processes each sample
- **Scorer** — how to grade the response

If your benchmark needs extra packages (like `datasets` for HuggingFace), include a `requirements.txt` — the container installs dependencies automatically.

In [ ]:
os.makedirs("benchmarks", exist_ok=True)
with open("benchmarks/mmlu_pro.py", "w") as f:
    f.write("""
from inspect_ai import Task, task
from inspect_ai.dataset import Sample, hf_dataset
from inspect_ai.scorer import choice
from inspect_ai.solver import multiple_choice, generate

# Convert HuggingFace record to Inspect AI format
def record_to_sample(record):
    return Sample(input=record["question"], target=record["answer"], choices=record["options"])

# Define the task: dataset + solver + scorer
@task
def mmlu_pro():
    dataset = hf_dataset("TIGER-Lab/MMLU-Pro", split="test", sample_fields=record_to_sample)
    return Task(dataset=dataset, solver=[multiple_choice(), generate()], scorer=choice())
""")
with open("benchmarks/requirements.txt", "w") as f:
    f.write("datasets>=2.21.0\n")
for root, _, files in os.walk("benchmarks"):
    for fn in files:
        s3.upload_file(os.path.join(root, fn), S3_BUCKET, os.path.join(root, fn))
print("\u2713 Benchmarks uploaded")

### Write the Eval Config

The YAML config tells the container which model to call, which benchmarks to run, and where to write results.

| Parameter | Value | Why |
|-----------|-------|-----|
| `max_connections` | 4 | Parallel requests to Bedrock |
| `max_retries` | 3 | Handle transient 503/429 errors |
| `temperature` | 0.0 | Deterministic, reproducible results |
| `max_tokens` | 4096 | Maximum output tokens per response |

In [ ]:
config = {
    "inference_provider": {"bedrock": {"model_id": "us.amazon.nova-2-lite-v1:0", "region": REGION}},
    "benchmarks": {"s3_path": f"s3://{S3_BUCKET}/benchmarks/", "tasks": [{"name": "mmlu_pro", "limit": 5}]},
    "eval": {"max_connections": 4, "max_retries": 3, "timeout": 600, "decoding": {"temperature": 0.0, "max_tokens": 4096}},
    "output": {"s3_path": f"s3://{S3_BUCKET}/eval-results/"},
}
os.makedirs("config", exist_ok=True)
with open("config/inspect_config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)
s3.upload_file("config/inspect_config.yaml", S3_BUCKET, "config/inspect_config.yaml")
print("\u2713 Config uploaded")
print("---")
print(yaml.dump(config, default_flow_style=False))

### (Optional) Enable MLflow Tracking

#### What is MLflow?

[Amazon SageMaker MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) is an experiment tracking service that lets you compare evaluation results across multiple runs. When enabled, the container automatically logs:

- **Metrics** — accuracy scores for each benchmark
- **Parameters** — endpoint name, task name, model configuration
- **Artifacts** — the full .eval log files for detailed analysis

#### Why use it?

- Compare model performance across training checkpoints
- Track improvement over fine-tuning iterations
- Share results with your team via the MLflow UI

#### How it works

1. You create a tracking server (one-time setup, takes ~5 min)
2. Add the server ARN to your eval config
3. The container logs metrics after each benchmark completes
4. View results in the MLflow web UI


In [ ]:
# Create MLflow tracking server (skip if you already have one)
mlflow_name = "inspectlens-tracker"
try:
    sagemaker_client = boto3.client("sagemaker", region_name=REGION)
    sagemaker_client.create_mlflow_tracking_server(
        TrackingServerName=mlflow_name,
        ArtifactStoreUri=f"s3://{S3_BUCKET}/mlflow/",
        RoleArn=ROLE_ARN,
    )
    print(f"\u2713 Creating MLflow server: {mlflow_name} (takes 5-10 min)")
except sagemaker_client.exceptions.ClientError as e:
    if "already exists" in str(e) or "ResourceInUse" in str(e):
        print(f"\u2713 MLflow server exists: {mlflow_name}")
    else:
        print(f"\u26a0 MLflow creation failed: {e}")

# Get ARN and add to config
try:
    resp = sagemaker_client.describe_mlflow_tracking_server(TrackingServerName=mlflow_name)
    MLFLOW_ARN = resp["TrackingServerArn"]
    config["tracking"] = {
        "mlflow_tracking_arn": MLFLOW_ARN,
        "mlflow_experiment_name": "nova-eval",
        "mlflow_tracing": True,
        "mlflow_log_artifacts": True,
    }
    # Re-upload config with MLflow
    with open("config/inspect_config.yaml", "w") as f:
        yaml.dump(config, f, default_flow_style=False)
    s3.upload_file("config/inspect_config.yaml", S3_BUCKET, "config/inspect_config.yaml")
    print(f"\u2713 Config updated with MLflow tracking")
except Exception as e:
    print(f"\u26a0 Skipping MLflow: {e}")


### View MLflow Results (after eval completes)

Run this cell after the evaluation job finishes to get the MLflow UI link:


In [ ]:
# Get MLflow UI URL
try:
    resp = sagemaker_client.create_presigned_mlflow_tracking_server_url(
        TrackingServerName=mlflow_name,
    )
    print(f"MLflow UI: {resp['AuthorizedUrl']}")
except Exception as e:
    print(f"MLflow URL not available: {e}")


### Submit the Evaluation Job

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import InputData, Compute
from sagemaker.core.shapes.shapes import StoppingCondition, OutputDataConfig

trainer = ModelTrainer(
    training_image=IMAGE_URI, role=ROLE_ARN,
    compute=Compute(instance_type="ml.m5.large", instance_count=1, volume_size_in_gb=30),
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{S3_BUCKET}/output/"),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=86400),
    base_job_name="inspect-eval-bedrock",
)
trainer.train(input_data_config=[InputData(channel_name="config", data_source=f"s3://{S3_BUCKET}/config/")], wait=False)
job_name = trainer._latest_training_job.training_job_name
print(f"\u2713 Job submitted: {job_name}")
print(f"\nMonitor in console:")
print(f"  https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/jobs/{job_name}")

### Monitor Progress

The job typically takes 2-5 minutes for small benchmarks.

> **If the job fails**, check:
> - Bedrock model access is enabled for your region
> - The IAM role has `AmazonBedrockFullAccess`
> - You have quota for `ml.m5.large` training instances

In [ ]:
sagemaker_client = boto3.client("sagemaker", region_name=REGION)
while True:
    resp = sagemaker_client.describe_training_job(TrainingJobName=job_name)
    status = resp["TrainingJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        print(f"\n\u2713 Job finished: {status}")
        if status == "Failed":
            print(f"  Reason: {resp.get('FailureReason')}")
        break
    print(f"  {status} / {resp.get('SecondaryStatus', '')}...", end="\r")
    time.sleep(15)

### View Results

Download results from S3 and view the status summary. You can also explore results interactively:

- **Inspect AI Viewer** (from S3 directly): `inspect view --log-dir s3://YOUR_BUCKET/eval-results/JOB_NAME/eval_results/`
- **Inspect AI Viewer** (local): `inspect view --log-dir ./results/`
- **VS Code Extension**: Install the [Inspect AI extension](https://marketplace.visualstudio.com/items?itemName=aisi-inspect.inspect-ai) to browse results in your editor


In [ ]:
os.makedirs("results", exist_ok=True)
resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=f"eval-results/{job_name}/eval_results/")
if "Contents" in resp:
    for obj in resp["Contents"]:
        s3.download_file(S3_BUCKET, obj["Key"], os.path.join("results", os.path.basename(obj["Key"])))
        print(f"  Downloaded: {os.path.basename(obj['Key'])}")

with open("results/_status.json") as f:
    st = json.load(f)
print(f"\nStatus: {st['status']}")
print(f"Passed: {st['passed_tasks']}")
print(f"Failed: {st['failed_tasks']}")

---

## Customization

### Change the model

```yaml
inference_provider:
  bedrock:
    model_id: "us.amazon.nova-lite-v1:0"     # Nova Lite
    model_id: "us.amazon.nova-pro-v1:0"      # Nova Pro
```

### Run more samples

Remove `limit` or increase it. For the full MMLU-Pro dataset (~12K samples):
```yaml
tasks:
  - name: mmlu_pro
```

### Add multiple benchmarks

```yaml
tasks:
  - name: mmlu_pro
  - name: boolq
  - name: arc_challenge
```

### Export results as CSV

```yaml
output:
  s3_path: "s3://your-bucket/eval-results/"
  output_format: "csv"
```

## Cleanup

In [ ]:
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=S3_BUCKET):
    if "Contents" in page:
        s3.delete_objects(Bucket=S3_BUCKET, Delete={"Objects": [{"Key": o["Key"]} for o in page["Contents"]]})
s3.delete_bucket(Bucket=S3_BUCKET)
for arn in policies:
    iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=arn)
iam.delete_role(RoleName=ROLE_NAME)
import shutil
for d in ["benchmarks", "config", "results"]:
    shutil.rmtree(d, ignore_errors=True)
print("\u2713 Cleaned up")